# الشبكات العصبية المتكررة

في الوحدة السابقة، تناولنا تمثيلات دلالية غنية للنص. الهيكلية التي كنا نستخدمها تلتقط المعنى المجمع للكلمات في الجملة، لكنها لا تأخذ في الاعتبار **ترتيب** الكلمات، لأن عملية التجميع التي تلي التضمينات تزيل هذه المعلومة من النص الأصلي. نظرًا لأن هذه النماذج غير قادرة على تمثيل ترتيب الكلمات، فهي لا تستطيع حل المهام الأكثر تعقيدًا أو الغموض مثل توليد النص أو الإجابة على الأسئلة.

لالتقاط معنى تسلسل النص، سنستخدم بنية شبكة عصبية تُسمى **الشبكة العصبية المتكررة**، أو RNN. عند استخدام RNN، نمرر جملتنا عبر الشبكة رمزًا واحدًا في كل مرة، وتنتج الشبكة بعض **الحالة**، التي نمررها مرة أخرى إلى الشبكة مع الرمز التالي.

![صورة تظهر مثالًا لتوليد شبكة عصبية متكررة.](../../../../../translated_images/ar/rnn.27f5c29c53d727b5.webp)

بالنظر إلى تسلسل الرموز $X_0,\dots,X_n$، تنشئ شبكة RNN تسلسلاً من كتل الشبكة العصبية، وتدرب هذا التسلسل من البداية للنهاية باستخدام الانتشار العكسي. تأخذ كل كتلة شبكة زوجًا $(X_i,S_i)$ كمدخل، وتنتج $S_{i+1}$ كنتيجة. الحالة النهائية $S_n$ أو الإخراج $Y_n$ تدخل في مصنف خطي لإنتاج النتيجة. جميع كتل الشبكة تشترك في نفس الأوزان، ويتم تدريبها نهاية لنهاية باستخدام تمرير واحد للانتشار العكسي.

> الشكل أعلاه يعرض الشبكة العصبية المتكررة في شكل غير مطوي (على اليسار)، وفي تمثيل متكرر أكثر تماسكًا (على اليمين). من المهم إدراك أن جميع خلايا RNN تمتلك نفس **الأوزان القابلة للمشاركة**.

لأن متجهات الحالة $S_0,\dots,S_n$ تمر عبر الشبكة، يمكن لـ RNN تعلم التبعيات التسلسلية بين الكلمات. على سبيل المثال، عندما تظهر كلمة *not* في مكان ما بالتسلسل، يمكنها تعلم نفي بعض العناصر داخل متجه الحالة.

داخليًا، تحتوي كل خلية RNN على مصفوفتين للأوزان: $W_H$ و $W_I$، والتحيز $b$. في كل خطوة RNN، بالنظر إلى المدخل $X_i$ وحالة الإدخال $S_i$، يتم حساب حالة الإخراج كـ $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$، حيث $f$ هي دالة التنشيط (غالبًا $\tanh$).

> للمشاكل مثل توليد النص (التي سنغطيها في الوحدة التالية) أو الترجمة الآلية، نريد أيضًا الحصول على قيمة إخراج في كل خطوة من خطوات RNN. في هذه الحالة، هناك مصفوفة أخرى $W_O$، ويتم حساب الإخراج كـ $Y_i=f(W_O\times S_i+b_O)$.

لنرى كيف يمكن أن تساعدنا الشبكات العصبية المتكررة في تصنيف مجموعة بيانات الأخبار الخاصة بنا.

> لبيئة الصندوق الرمل، نحتاج إلى تشغيل الخلية التالية للتأكد من تثبيت المكتبة المطلوبة، وتحميل البيانات مسبقًا. إذا كنت تعمل محليًا، يمكنك تخطي الخلية التالية.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

عند تدريب النماذج الكبيرة، قد تصبح تخصيص ذاكرة وحدة معالجة الرسومات (GPU) مشكلة. قد نحتاج أيضًا إلى تجربة أحجام دفعات صغيرة مختلفة، بحيث تتناسب البيانات مع ذاكرة وحدة معالجة الرسومات الخاصة بنا، مع أن يكون التدريب سريعًا بما فيه الكفاية. إذا كنت تشغل هذا الكود على جهازك الخاص المزود بوحدة معالجة الرسومات، يمكنك تجربة تعديل حجم الدفعة الصغيرة لتسريع عملية التدريب.

> **ملاحظة**: بعض إصدارات تعريفات NVidia معروفة بعدم تحرير الذاكرة بعد تدريب النموذج. نحن نشغل عدة أمثلة في هذه المفكرة، وقد يسبب ذلك نفاد الذاكرة في بعض الإعدادات، خاصة إذا كنت تجري تجاربك الخاصة كجزء من نفس المفكرة. إذا واجهت أخطاء غريبة عند بدء تدريب النموذج، فقد ترغب في إعادة تشغيل نواة المفكرة.


In [3]:
batch_size = 16
embed_size = 64

## مُصنّف RNN بسيط

في حالة RNN البسيط، كل وحدة متكررة هي شبكة خطية بسيطة، تأخذ متجه إدخال ومتجه حالة، وتنتج متجه حالة جديد. في Keras، يمكن تمثيل ذلك بواسطة طبقة `SimpleRNN`.

بينما يمكننا تمرير رموز مشفرة بطريقة الواحد الساخن مباشرةً إلى طبقة RNN، إلا أن هذا ليس فكرة جيدة بسبب أبعادها العالية. لذلك، سنستخدم طبقة تضمين لتقليل أبعاد متجهات الكلمات، تليها طبقة RNN، وأخيرًا مُصنّف `Dense`.

> **ملاحظة**: في الحالات التي لا تكون الأبعاد فيها عالية جدًا، مثل عند استخدام ترميز الرموز على مستوى الحروف، قد يكون من المنطقي تمرير الرموز المشفرة بطريقة الواحد الساخن مباشرةً إلى خلية RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **ملاحظة:** نستخدم هنا طبقة تضمين غير مدربة للبساطة، ولكن للحصول على نتائج أفضل يمكننا استخدام طبقة تضمين مدربة مسبقًا باستخدام Word2Vec، كما وُصف في الوحدة السابقة. سيكون من الجيد أن تقوم بتكييف هذا الكود ليعمل مع التضمينات المدربة مسبقًا.  

الآن دعونا ندرب شبكة RNN الخاصة بنا. بشكل عام، من الصعب تدريب شبكات RNN، لأنه بمجرد فك خلايا RNN على طول طول التسلسل، يكون عدد الطبقات الناتجة المتضمنة في الانتشار العكسي كبيرًا جدًا. لذا نحتاج إلى اختيار معدل تعلم أصغر، وتدريب الشبكة على مجموعة بيانات أكبر لإنتاج نتائج جيدة. يمكن أن يستغرق هذا وقتًا طويلاً، لذا يُفضل استخدام GPU.  

لتسريع الأمور، سنقوم بتدريب نموذج RNN فقط على عناوين الأخبار، متجاهلين الوصف. يمكنك تجربة التدريب مع الوصف ومعرفة ما إذا كان بإمكانك جعل النموذج يتدرب.  


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **ملاحظة** من المحتمل أن تكون الدقة أقل هنا، لأننا ندرب فقط على عناوين الأخبار.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


الآن بعد أن بدأنا في استخدام الإخفاء، يمكننا تدريب النموذج على مجموعة البيانات الكاملة من العناوين والوصفات.

> **ملاحظة**: هل لاحظت أننا كنا نستخدم محول متجهات تم تدريبه على عناوين الأخبار، وليس على كامل نص المقالة؟ من المحتمل أن يؤدي هذا إلى تجاهل بعض الرموز، لذلك من الأفضل إعادة تدريب محول المتجهات. ومع ذلك، قد يكون لهذا تأثير صغير جداً، لذا سنلتزم بالمحول المتجهات المدرب مسبقًا السابق من أجل البساطة.


## LSTM: الذاكرة طويلة وقصيرة المدى  

واحدة من المشاكل الرئيسية في الشبكات العصبية المتكررة (RNNs) هي **تلاشي التدرجات**. يمكن أن تكون الشبكات العصبية المتكررة طويلة جدًا، وقد تواجه صعوبة في نقل التدرجات إلى الوراء حتى الطبقة الأولى من الشبكة أثناء عملية الانحدار العكسي. عندما يحدث هذا، لا تستطيع الشبكة تعلم العلاقات بين الرموز البعيدة. إحدى طرق تجنب هذه المشكلة هي إدخال **إدارة صريحة للحالة** باستخدام **البوابات**. أكثر البنى شيوعًا التي تقدم البوابات هي **الذاكرة طويلة وقصيرة المدى** (LSTM) و **وحدة الترحيل المقيدة** (GRU). سنغطي LSTM هنا.  

![صورة تظهر مثالاً لخلية ذاكرة طويلة وقصيرة المدى](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)  

تُنظم شبكة LSTM بطريقة مشابهة لشبكة RNN، لكن هناك حالتين يتم تمريرهما من طبقة إلى أخرى: الحالة الفعلية $c$، والمتجه المخفي $h$. في كل وحدة، يتم دمج المتجه المخفي $h_{t-1}$ مع الإدخال $x_t$، ويتحكمان معًا في ما يحدث للحالة $c_t$ والمخرجات $h_{t}$ من خلال **البوابات**. كل بوابة لها تفعيل سيجمويد (الناتج في النطاق $[0,1]$)، والذي يمكن اعتباره قناعًا ثنائيًا عند ضربه في متجه الحالة. لدى LSTM البوابات التالية (من اليسار إلى اليمين في الصورة أعلاه):  
* **بوابة النسيان** التي تحدد أي مكونات من المتجه $c_{t-1}$ نحتاج إلى نسيانها وأيها نمررها.  
* **بوابة الإدخال** التي تحدد مقدار المعلومات من متجه الإدخال والمتجه المخفي السابق التي يجب دمجها في متجه الحالة.  
* **بوابة الإخراج** التي تأخذ متجه الحالة الجديد وتقرر أي مكوناته سيتم استخدامها لإنتاج المتجه المخفي الجديد $h_t$.  

يمكن التفكير في مكونات الحالة $c$ على أنها أعلام يمكن تشغيلها وإيقافها. على سبيل المثال، عندما نواجه اسم *Alice* في التسلسل، نفترض أنه يشير إلى امرأة، ونرفع العلم في الحالة الذي يشير إلى وجود اسم مؤنث في الجملة. وعندما نواجه بعد ذلك الكلمات *and Tom*، نرفع العلم الذي يشير إلى وجود اسم جمع. وهكذا، من خلال التلاعب بالحالة يمكننا تتبع الخصائص النحوية للجملة.  

> **ملاحظة**: إليك مصدر رائع لفهم البنية الداخلية لـ LSTMs: [فهم شبكات LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) بواسطة كريستوفر أولاه.  

بينما قد تبدو البنية الداخلية لخلية LSTM معقدة، يخفي Keras هذا التنفيذ داخل طبقة `LSTM`، لذا فإن الشيء الوحيد الذي نحتاج إلى فعله في المثال أعلاه هو استبدال الطبقة المتكررة:  


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **ملاحظة** أن تدريب LSTMs بطيء جداً أيضاً، وقد لا تلاحظ زيادة كبيرة في الدقة في بداية التدريب. قد تحتاج إلى الاستمرار في التدريب لبعض الوقت لتحقيق دقة جيدة.


## الشبكات العصبية التكرارية ثنائية الاتجاه ومتعددة الطبقات

في أمثلتنا حتى الآن، تعمل الشبكات التكرارية من بداية التسلسل حتى نهايته. هذا يبدو طبيعيًا بالنسبة لنا لأنه يتبع نفس الاتجاه الذي نقرأ فيه أو نستمع إلى الكلام. ومع ذلك، في السيناريوهات التي تتطلب وصولًا عشوائيًا لتسلسل الإدخال، من المنطقي تشغيل الحساب التكراري في كلا الاتجاهين. تُسمى الشبكات العصبية التكرارية التي تسمح بالحساب في كلا الاتجاهين **شبكات عصبية تكرارية ثنائية الاتجاه**، ويمكن إنشاؤها بتغليف الطبقة التكرارية بطبقة خاصة `Bidirectonal`.

> **ملاحظة**: تقوم طبقة `Bidirectional` بعمل نسختين من الطبقة بداخلها، وتضبط خاصية `go_backwards` في إحدى هاتين النسختين إلى `True`، مما يجعلها تسير عكس الاتجاه على طول التسلسل.

تلتقط الشبكات العصبية التكرارية، سواء كانت أحادية الاتجاه أو ثنائية الاتجاه، أنماطًا داخل التسلسل، وتخزنها في متجهات الحالة أو تعيدها كمخرجات. كما هو الحال مع الشبكات التلافيفية، يمكننا بناء طبقة تكرارية أخرى تلي الطبقة الأولى لالتقاط أنماط على مستوى أعلى، مبنية من الأنماط على مستوى أدنى التي استخرجتها الطبقة الأولى. يقودنا هذا إلى مفهوم **الشبكة العصبية التكرارية متعددة الطبقات**، والتي تتكون من شبكتين تكراريتين أو أكثر، حيث تُمرر مخرجات الطبقة السابقة إلى الطبقة التالية كمدخلات.

![صورة توضح شبكة RNN متعددة الطبقات طويلة قصيرة المدى](../../../../../translated_images/ar/multi-layer-lstm.dd975e29bb2a59fe.webp)

*صورة من [هذا المنشور الرائع](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) بواسطة فرناندو لوبيز.*

يجعل Keras بناء هذه الشبكات مهمة سهلة، لأنك تحتاج فقط إلى إضافة المزيد من الطبقات التكرارية إلى النموذج. بالنسبة لجميع الطبقات باستثناء الأخيرة، نحتاج إلى تحديد معامل `return_sequences=True`، لأننا نحتاج إلى أن تعيد الطبقة جميع الحالات الوسيطة، وليس فقط الحالة النهائية للحساب التكراري.

لننشئ شبكة LSTM ثنائية الاتجاه بطبقتين لمشكلة التصنيف لدينا.

> **ملاحظة** هذا الكود مرة أخرى يستغرق وقتًا طويلاً للإكمال، لكنه يعطينا أعلى دقة رأيناها حتى الآن. لذلك ربما يستحق الانتظار ورؤية النتيجة.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## الشبكات العصبونية المتكررة لمهام أخرى

حتى الآن، ركزنا على استخدام الشبكات العصبونية المتكررة لتصنيف تسلسلات النصوص. لكنها يمكن أن تتعامل مع مهام كثيرة أخرى، مثل توليد النصوص والترجمة الآلية &mdash; سنناقش تلك المهام في الوحدة التالية.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
